In [ ]:
import json
import platform
from datetime import datetime, timezone

environment = {
    "run_id": "colab_environment_check",
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "pytorch_version": None,
    "pytorch_cuda_version": None,
    "cuda_available": False,
    "gpus": [],
    "gpu_check_passed": False,
    "diagnostics": [],
}

try:
    import torch

    environment["pytorch_version"] = str(torch.__version__)
    environment["pytorch_cuda_version"] = torch.version.cuda
    environment["cuda_available"] = torch.cuda.is_available()

    if environment["cuda_available"]:
        for index in range(torch.cuda.device_count()):
            gpu = torch.cuda.get_device_properties(index)
            environment["gpus"].append({
                "index": index,
                "name": gpu.name,
                "total_vram_gib": round(gpu.total_memory / 1024**3, 2),
            })

        environment["gpu_check_passed"] = bool(environment["gpus"])
    else:
        environment["diagnostics"].append(
            "Chưa truy cập được GPU. Hãy kiểm tra loại runtime."
        )

except (ImportError, RuntimeError, OSError) as error:
    environment["diagnostics"].append(str(error))

print(json.dumps(environment, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path
from google.colab import files

if "environment" not in globals():
    raise RuntimeError("Lỗi: Chưa có dữ liệu môi trường.")

report_path = Path("/content/outputs/colab_environment_check/config.json")
report_path.parent.mkdir(parents=True, exist_ok=True)

report_path.write_text(
    json.dumps(environment, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

files.download(str(report_path))